# OpenKind: unified speed, accuracy and validity lab
**Version 3.0.0 · T4/L4 GPU plus a Go CPU comparator · one notebook**

Continue from the completed 9B/T4 experiment, using the supplied **working paper revision 0.8**. Test the missing joins: routing dependency rules, selective/grouped readouts, joint output probabilities, process isolation, actual Q/K/input-length costs, and a compact **indecis** alternative.

This notebook contains the full runner, prior native-build recovery, the Go measurement adapter, source tests, and an explicit question-coverage ledger. **There are no prefilled new model results.** Completion means the selected bounded experiments ran; it does not mean every scientific gate passed.

Select **T4 or L4**, then **Runtime → Run all**. Keep `RESUME_RUN` blank for a new run. Retain the existing local cache to permit verified binary/model reuse. A new VM may still need native compilation.

**The scope is not universal:** a Colab CPU/GPU study cannot validate the separate Mac/MLX runtime or independent human review. Original Qwen document-retention, matched model-size/code-head, expert-streaming, diffusion and RL experiments remain explicitly unmeasured rather than receiving an invented answer.

## 1. Controls
All runnable tracks are enabled by default. The session budget is an allowed-work ceiling, **not an estimate of completion time**. A partial run can continue under exactly the same identity; interrupted history blocks restart whole processes. The old completed E40 run must never be reused.

The normal GPU phase holds one 9B model process at a time. Only the **dedicated-process isolation treatment** intentionally attempts two resident copies, with actual memory admission and full-offload checks. It does not silently fall back to CPU or a smaller model.

In [1]:
CPU_VALIDATE_ONLY = False #@param {type:"boolean"}
RUN_NATIVE = True #@param {type:"boolean"}
RUN_INDECIS = True #@param {type:"boolean"}
RUN_PUBLIC_NLI = True #@param {type:"boolean"}
RUN_SCALING = True #@param {type:"boolean"}
RUN_ROBUSTNESS = True #@param {type:"boolean"}
RUN_INDECIS_RETENTION = True #@param {type:"boolean"}
RUN_INDECIS_INT8 = True #@param {type:"boolean"}
SESSION_MINUTES = 300 #@param {type:"integer"}
BUILD_TIMEOUT_MINUTES = 120 #@param {type:"integer"}
BUILD_JOBS = 0 #@param {type:"integer"}
RESUME_RUN = "" #@param {type:"string"}
WORK_CACHE = "/content/openkind9b_t4l4_cache"
DRIVE_LAB = "/content/drive/MyDrive/Colab Notebooks/OpenKind_MoE_Lab"

## 2. Restore the pinned, self-contained source bundle
The embedded source is hash-checked before import. Existing notebook modules are unloaded so an earlier runner cannot remain silently active. The downloaded model and native compiler cache are separate from this source extraction.

In [2]:
import base64, hashlib, importlib, io, json, pathlib, sys, zipfile
BUNDLE_SHA256 = '50fc096d6d1c02ca39a6b4b3bd822fd5a3a809eed9eb6f6ea291122bfb890db0'
BUNDLE_BASE64 = '...'
raw = base64.b64decode(BUNDLE_BASE64)
assert hashlib.sha256(raw).hexdigest() == BUNDLE_SHA256, "Embedded bundle mismatch"
SOURCE_ROOT = pathlib.Path('/tmp/openkind_unified_sources') / BUNDLE_SHA256[:20]
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for member in z.infolist():
        p = pathlib.PurePosixPath(member.filename)
        assert not p.is_absolute() and '..' not in p.parts, 'Unsafe archive path'
    z.extractall(SOURCE_ROOT)
source_manifest = json.loads((SOURCE_ROOT/'SOURCE_MANIFEST.json').read_text())
for name, expected in source_manifest.items():
    assert hashlib.sha256((SOURCE_ROOT/name).read_bytes()).hexdigest() == expected, name
for name in list(sys.modules):
    if name.startswith(('openkind9b_', 'unified_', 'e39_inherited')):
        del sys.modules[name]
sys.path.insert(0, str(SOURCE_ROOT)); importlib.invalidate_caches()
print('Source bundle verified:', BUNDLE_SHA256)
print('Working paper SHA-256:', source_manifest['assets/WORKING_PAPER_R08.md'])

Source bundle verified: 50fc096d6d1c02ca39a6b4b3bd822fd5a3a809eed9eb6f6ea291122bfb890db0
Working paper SHA-256: ed2b158ec74283c9dd236e2a3e08bbcbfb18711d104dad51ac81133f1db967a2


## 3. Dependencies and CPU contract tests
Existing Python package versions are retained; only missing packages are installed. The run records actual versions. No GPU model, pretrained indecis model, or training is exercised by these tests. The actual pinned Go compiler and upstream project are prepared later, with separate receipts.

In [3]:
import importlib.util, subprocess, os
requirements = {'numpy':'numpy','pandas':'pandas','matplotlib':'matplotlib',
                'huggingface_hub':'huggingface_hub','tokenizers':'tokenizers',
                'pyarrow':'pyarrow','tabulate':'tabulate'}
missing = [package for module,package in requirements.items()
           if importlib.util.find_spec(module) is None]
if missing and not CPU_VALIDATE_ONLY:
    subprocess.run([sys.executable,'-m','pip','install','--quiet',*missing],check=True)
# Bounded network waits; never print HF_TOKEN or other credentials.
os.environ.setdefault('HF_HUB_ETAG_TIMEOUT','30')
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT','120')
log = SOURCE_ROOT/'CPU_VALIDATION.log'
with log.open('w') as f:
    check = subprocess.run([sys.executable,'-m','unittest','discover','-p','test_*.py'],
                           cwd=SOURCE_ROOT,stdout=f,stderr=subprocess.STDOUT,timeout=180)
print(log.read_text()[-3500:])
assert check.returncode == 0, f'CPU checks failed; inspect {log}'
print('CPU contract tests passed. Native CUDA and real indecis inference remain separate checks.')

k 2/39 — Q4_K_M history
Completed: history Q4_K_M  noop
Block 3/39 — Q4_K_M history
Completed: history Q4_K_M  native
Block 4/39 — Q4_K_M history
Completed: history Q4_K_M  json
Block 5/39 — Q4_K_M history
Completed: history Q4_K_M  noop
Block 6/39 — Q4_K_M history
Completed: history Q4_K_M  native
Block 7/39 — Q4_K_M history
Completed: history Q4_K_M  json
Block 8/39 — Q4_K_M cold_trace
Completed: cold_trace Q4_K_M  
Block 9/39 — Q4_K_M cold_trace
Completed: cold_trace Q4_K_M  
Block 10/39 — Q4_K_M cold_trace
Completed: cold_trace Q4_K_M  
Block 11/39 — Q4_K_M cold_trace
Completed: cold_trace Q4_K_M  
Block 12/39 — Q4_K_M quality
Completed: quality Q4_K_M A 
Block 13/39 — Q4_K_M quality
Completed: quality Q4_K_M B 
Block 14/39 — Q4_K_M quality
Completed: quality Q4_K_M C 
Block 15/39 — Q4_K_M quality
Completed: quality Q4_K_M D 
Block 16/39 — Q4_K_M quality
Completed: quality Q4_K_M E 
Block 17/39 — Q4_K_M quality
Completed: quality Q4_K_M E 
Block 18/39 — Q4_K_M quality
Completed: qu

## 4. Register the exact questions and requested comparisons
The ledger distinguishes a measured outcome from an unmeasured question. A fast arm with failed execution parity is not an interchangeable optimization. A changed model is evaluated for its own quality, not forced to reproduce another model's mistakes.

**GPU arms:** D reference; R timed route-priority rule; X targeted indexes; G three-call grouping; GR grouping plus route rule; J joint routing/urgency codes; E isolated generated JSON. R/GR have point decisions and probability bounds, not fabricated joint distributions. J preserves exact derived marginals from a declared joint output.

**Compact arms:** frozen open embedding scorer, trained fixed heads, trained open scorer, matched compact policy-only/replay continuations, and hardware-qualified INT8/assembly diagnostics. Fixed heads do not answer arbitrary newly described questions merely because the API has a schema.

In [4]:
import copy, pandas as pd
import unified_runner as lab
import openkind9b_core as core
CONFIG = copy.deepcopy(lab.DEFAULTS)
CONFIG.update(run_native=RUN_NATIVE,run_indecis=RUN_INDECIS,run_public_nli=RUN_PUBLIC_NLI,
              run_scaling=RUN_SCALING,run_robustness=RUN_ROBUSTNESS,
              run_indecis_retention=RUN_INDECIS_RETENTION,run_indecis_int8=RUN_INDECIS_INT8)
assert RUN_NATIVE or RUN_INDECIS or CPU_VALIDATE_ONLY, 'Select a real track'
display(pd.DataFrame(lab.QUESTIONS,columns=['ID','Question','Track','Boundary']))
print('Protocol:',lab.PROTOCOL)
print('Native schema arms:',CONFIG['native_arms'])

,ID,Question,Track,Boundary
0,Q01,Route-priority composition improves measured f...,quality,R versus D; point-only probability boundary
1,Q02,Grouped or targeted indexed fields recover cas...,quality,"X/G/GR/J versus D, fresh templates and regress..."
2,Q03,Exact joint probabilities permit consistent de...,quality,J joint categorical source; no marginal-indepe...
3,Q04,Dedicated processes remove mixed JSON/native h...,isolation,failing mixed control plus dual-resident treat...
4,Q05,Fresh process after JSON removes drift at an a...,isolation,matched restart treatment including startup/pr...
5,Q06,Quality remains stable across same-process nat...,isolation,native-only reference plus fresh-reset compari...
6,Q07,Static-prefix reuse and true first use remain ...,cache,E40-compatible cache pairs and fresh-process t...
7,Q08,Question/cardinality scaling has useful qualit...,scaling,Q/K/actual input-token grids; synthetic mechan...
8,Q09,"Question addition, key names, and code mapping...",robustness,separate transformations; no false opaque-key ...
9,Q10,A small Go encoder is a useful accuracy/resour...,indecis_quality,fixed trained vs frozen/fitted open scorer; di...


Protocol: openkind-unified-decisions/v3.0.0
Native schema arms: ['D', 'R', 'X', 'G', 'GR', 'J', 'E']


## 5. Mount Drive and prepare source-separated inputs
New policy splits use 768/64/64/64/96 cases for fitting/development/calibration-fit/calibration-gate/confirmation. Confirmation uses withheld wording families; this is **authored template transfer**, not independent production validation. The old 96- and 36-case panels remain separate regression data and never enter new fitting.

The public SNLI slice uses original labels and disjoint premise groups, with source-test confirmation. It tests **short-premise NLI**, not the old Qwen ContractNLI/QASPER retention claim. All input and source identities are written before model measurements. No protected historical final partition is read.

In [5]:
from unified_data import all_local_cases
DATA_RECEIPT = {}
if CPU_VALIDATE_ONLY:
    DATA = all_local_cases()
    print('CPU-only validation: no Drive, public-data download, model download or inference.')
else:
    from google.colab import drive
    if not pathlib.Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
    if RESUME_RUN:
        old = pathlib.Path(RESUME_RUN)
        assert (old/'manifest.json').is_file(), 'Resume requires an existing incomplete run'
        assert not (old/'COMPLETE.json').exists(), 'A completed run ID is immutable'
        DATA = core.load_json(old/'dataset.json')
        DATA_RECEIPT = core.load_json(old/'data_preparation.json')
    else:
        # A source-access error is not replaced by invented NLI rows.
        DATA,DATA_RECEIPT = lab.prepare_data(WORK_CACHE,public=RUN_PUBLIC_NLI)
    print('Prepared inputs:',len(DATA))
display(pd.DataFrame(DATA).groupby(['task','split','panel']).size().rename('cases').to_frame())

Mounted at /content/drive
CPU checks: all 144 gold labels independently reconstructed; all output mappings preserve semantics.


plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  413kB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  412kB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Prepared inputs: 1860


cases
task   split            panel                              
nli    calibration_fit  public_snli_calibration_fit      48
       calibration_gate public_snli_calibration_gate     48
       confirmation     public_snli_confirmation         96
       development      public_snli_development          48
       train            public_snli_train               384
policy calibration      historical_calibration           48
       calibration_fit  authored_calibration_fit         64
       calibration_gate authored_calibration_gate        64
       confirmation     fresh_template_transfer          96
       development      authored_development             64
       diagnostic       historical_e40_36                36
       test             historical_e39_96                96
       train            authored_train                  768

## 6. Create a new evidence identity—or strictly resume a partial one
The source, data, configuration and host identities are recorded. Each backend also receives its own immutable file/binary lock before measurement. Trained compact weights are saved separately from the evidence ZIP, and their hashes are required on resume.

In [6]:
STUDY = None
if not CPU_VALIDATE_ONLY:
    runtime = core.runtime_identity(gpu=RUN_NATIVE)
    runtime['cpu_count'] = os.cpu_count()
    cpuinfo = pathlib.Path('/proc/cpuinfo')
    if cpuinfo.exists():
        runtime['cpu_model'] = next((line.split(':',1)[1].strip() for line in cpuinfo.read_text().splitlines()
                                     if line.startswith('model name')), 'unknown')
    STUDY = core.Study(pathlib.Path(DRIVE_LAB)/'unified_decision_runs',WORK_CACHE,CONFIG,DATA,
                       minutes=SESSION_MINUTES,reserve_minutes=5,resume=RESUME_RUN,
                       pre_run_runtime=runtime)
    STUDY.build_options = {'timeout_minutes':BUILD_TIMEOUT_MINUTES,'jobs':BUILD_JOBS,
                          'legacy_recovery_from_run':''}
    STUDY.model_store = str(pathlib.Path(DRIVE_LAB)/'unified_decision_models')
    STUDY.write('data_preparation.json',DATA_RECEIPT)
    STUDY.write('notebook_bundle_identity.json',{'sha256':BUNDLE_SHA256,'protocol':lab.PROTOCOL})
    STUDY.write('resource_controls.json',{'session_minutes':SESSION_MINUTES,
                'build_timeout_minutes':BUILD_TIMEOUT_MINUTES,'build_jobs':BUILD_JOBS,
                'model_store':STUDY.model_store})
    (STUDY.folder/'EXPERIMENT_CONTRACT.md').write_text((SOURCE_ROOT/'EXPERIMENT_CONTRACT.md').read_text())
    (STUDY.folder/'CPU_VALIDATION.log').write_text(log.read_text())
    print('New/continued evidence:',STUDY.folder)
else:
    print('Journal creation skipped: no real execution identity or completion claim in CPU mode.')

Evidence folder: /content/drive/MyDrive/Colab Notebooks/OpenKind_MoE_Lab/unified_decision_runs/20260929T194506_694066Z_dc1f74
New/continued evidence: /content/drive/MyDrive/Colab Notebooks/OpenKind_MoE_Lab/unified_decision_runs/20260929T194506_694066Z_dc1f74


## 7. Execute the selected experiments
The notebook first validates the inherited native backend and every new schema's actual token paths. It then runs matched isolation traces, panel-sized quality blocks, cache/cold-start/memory probes and the scaling/robustness matrices. The compact phase compiles and runs **actual indecis**, trains only on declared fitting rows and evaluates raw and separately gated calibration.

Every block writes raw inputs/replies, actual model and binary identity, selected answers, full available distributions, timings and scope. Independent failures remain visible; they do not turn into zero latency or silently chosen fallback models. Optional INT8 without the required CPU capability is explicitly not executed.

For a partial run, keep the same notebook, controls, VM/runtime and data, set `RESUME_RUN` to the printed directory, then run again. A completed ID cannot be reused. The session and command limits do not guarantee all experiments finish within one allocation.

In [7]:
OUTCOME = None
if CPU_VALIDATE_ONLY:
    print('SKIPPED: native build, real Go dependency build, pretrained downloads, training and all real measurements.')
else:
    try:
        OUTCOME = lab.run(STUDY)
    except BaseException as exc:
        import traceback
        STUDY.write('UNHANDLED_FAILURE.json',{'type':type(exc).__name__,'message':str(exc),
                    'traceback':traceback.format_exc(),'promotion':False})
        try: STUDY.finish('PARTIAL')
        except Exception as archive_exc: print('Archival error:',str(archive_exc))
        raise
    print('Execution:',OUTCOME['execution_status'])
    print('Completed blocks:',OUTCOME['completed_blocks'])
    print('Read scientific gates independently; model/service promotion remains false.')

GPU preflight: Tesla T4 15.0 GiB; ['Q4_K_M']
source_download.log: 0.0 min elapsed; log 0 bytes; last output: (compiler has not written output yet)
Creating a fresh native build tree; existing caches are retained.
audit_json_compile.log: 0.0 min elapsed; log 0 bytes; last output: (compiler has not written output yet)
audit_json_roundtrip.log: 0.0 min elapsed; log 0 bytes; last output: (compiler has not written output yet)
native_configure.log: 0.0 min elapsed; log 0 bytes; last output: (compiler has not written output yet)
native_build_control.log: 0.0 min elapsed; log 0 bytes; last output: (compiler has not written output yet)
native_build_control.log: 0.8 min elapsed; 4% targets; log 1,175 bytes; last output: [  4%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/gguf.cpp.o
native_build_control.log: 1.5 min elapsed; 8% targets; log 2,962 bytes; last output: [  8%] Building CXX object ggml/src/CMakeFiles/ggml-cpu.dir/ggml-cpu/iqp.cpp.o
native_build_control.log: 2.3 min elapsed; 1

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  212MB            

model.safetensors: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

Completed: indecis_quality  I_OPEN_FROZEN 
training.log: 0.0 min elapsed; log 0 bytes; last output: (compiler has not written output yet)
training.log: 0.8 min elapsed; log 26 bytes; last output: step 20/216 loss 5.360011
training.log: 1.5 min elapsed; log 52 bytes; last output: step 40/216 loss 4.971100
training.log: 2.3 min elapsed; log 78 bytes; last output: step 60/216 loss 5.308102
training.log: 3.0 min elapsed; log 131 bytes; last output: step 100/216 loss 1.013562
training.log: 3.8 min elapsed; log 158 bytes; last output: step 120/216 loss 0.997478
training.log: 4.5 min elapsed; log 185 bytes; last output: step 140/216 loss 0.910816
training.log: 5.3 min elapsed; log 212 bytes; last output: step 160/216 loss 0.924965
training.log: 6.0 min elapsed; log 266 bytes; last output: step 200/216 loss 0.723486
Completed: indecis_fit  I_FIXED 
training.log: 0.0 min elapsed; log 0 bytes; last output: (compiler has not written output yet)
training.log: 0.8 min elapsed; log 0 bytes; last out

## 8. Read what worked, what failed and what remains unmeasured
`quality_summary.csv` contains new-run observations only. `class_recall.csv` exposes suffix and class failures; `paired_comparisons.json` averages technical repeats before group resampling. `calibration_results.json` separates fitting, acceptance and confirmation, and `confidence_policies.json` retains review-all and error/coverage denominators.

Read `measurement_gates.json` for quality sentinels, cache, first-use, sampled memory, matched isolation and CPU execution changes. `scaling_validity.json` and `robustness_summary.json` retain observed probability and answer differences. A passing treatment without a reproduced failing control is not a repair. Point-only routing corrections must never be advertised with a fabricated probability vector. GPU HTTP and CPU IPC timings are different system profiles, not a neutral architecture ablation.

In [8]:
if OUTCOME is not None:
    display(pd.DataFrame(OUTCOME['questions']))
    if OUTCOME['quality']:
        display(pd.DataFrame(OUTCOME['quality'])[['arm','panel','unique_cases','field_accuracy',
                  'all_correct','request_p50_ms','probability_coverage']])
    display(pd.DataFrame(OUTCOME['isolation']))
    print('Evidence:',STUDY.folder)
    print('Archive:',STUDY.folder.with_suffix('.zip'))
    print('Report:',STUDY.folder/'REPORT.md')
    print('Completion marker:',(STUDY.folder/'COMPLETE.json').exists())
    print('RESUME_RUN =',repr(str(STUDY.folder)) if OUTCOME['execution_status']=='PARTIAL' else 'DO NOT REUSE COMPLETED ID')
else:
    print('No new scientific result exists in CPU-validation mode.')

,id,question,status,boundary
0,Q01,Route-priority composition improves measured f...,MEASURED_SEE_DETAILED_GATES,R versus D; point-only probability boundary
1,Q02,Grouped or targeted indexed fields recover cas...,MEASURED_SEE_DETAILED_GATES,"X/G/GR/J versus D, fresh templates and regress..."
2,Q03,Exact joint probabilities permit consistent de...,MEASURED_SEE_DETAILED_GATES,J joint categorical source; no marginal-indepe...
3,Q04,Dedicated processes remove mixed JSON/native h...,MEASURED_SEE_DETAILED_GATES,failing mixed control plus dual-resident treat...
4,Q05,Fresh process after JSON removes drift at an a...,MEASURED_SEE_DETAILED_GATES,matched restart treatment including startup/pr...
5,Q06,Quality remains stable across same-process nat...,MEASURED_SEE_DETAILED_GATES,native-only reference plus fresh-reset compari...
6,Q07,Static-prefix reuse and true first use remain ...,MEASURED_SEE_DETAILED_GATES,E40-compatible cache pairs and fresh-process t...
7,Q08,Question/cardinality scaling has useful qualit...,MEASURED_SEE_DETAILED_GATES,Q/K/actual input-token grids; synthetic mechan...
8,Q09,"Question addition, key names, and code mapping...",MEASURED_SEE_DETAILED_GATES,separate transformations; no false opaque-key ...
9,Q10,A small Go encoder is a useful accuracy/resour...,MEASURED_SEE_DETAILED_GATES,fixed trained vs frozen/fitted open scorer; di...


,arm,panel,unique_cases,field_accuracy,all_correct,request_p50_ms,probability_coverage
0,D,authored_calibration_fit,64,0.833333,0.265625,289.005176,1.0
1,D,authored_calibration_gate,64,0.843750,0.281250,290.019661,1.0
2,D,authored_development,64,0.828125,0.296875,290.225646,1.0
3,D,fresh_template_transfer,96,0.826389,0.239583,306.923924,1.0
4,D,historical_e39_96,96,0.861111,0.322917,288.332382,1.0
...,...,...,...,...,...,...,...
101,X,authored_calibration_gate,64,0.859375,0.328125,202.856653,1.0
102,X,authored_development,64,0.875000,0.468750,204.912100,1.0
103,X,fresh_template_transfer,96,0.921875,0.604167,222.217620,1.0
104,X,historical_e39_96,96,0.937500,0.625000,202.404113,1.0


,isolation,drop_action,contexts,passed,matched_control_failure_reproduced,repair_candidate,total_trace_ms,max_dp,field_flip_occurrences,scope
0,mixed,False,1,False,True,False,84720.491855,0.080189,0,A passing treatment without failing control is...
1,native_only,False,1,True,True,False,50478.490604,0.000000,0,A passing treatment without failing control is...
2,separate_process,False,1,True,True,True,89635.313882,0.000000,0,A passing treatment without failing control is...
3,restart_process,False,1,True,True,True,305114.154755,0.000000,0,A passing treatment without failing control is...
4,mixed,True,4,False,True,False,104453.415021,0.094815,3,A passing treatment without failing control is...
5,native_only,True,4,True,True,False,70832.971108,0.000000,0,A passing treatment without failing control is...
6,separate_process,True,4,True,True,True,108532.590254,0.000000,0,A passing treatment without failing control is...
7,restart_process,True,4,True,True,True,410283.186726,0.000000,0,A passing treatment without failing control is...


Evidence: /content/drive/MyDrive/Colab Notebooks/OpenKind_MoE_Lab/unified_decision_runs/20260929T194506_694066Z_dc1f74
Archive: /content/drive/MyDrive/Colab Notebooks/OpenKind_MoE_Lab/unified_decision_runs/20260929T194506_694066Z_dc1f74.zip
Report: /content/drive/MyDrive/Colab Notebooks/OpenKind_MoE_Lab/unified_decision_runs/20260929T194506_694066Z_dc1f74/REPORT.md
Completion marker: True
RESUME_RUN = DO NOT REUSE COMPLETED ID


## Evidence boundaries and references
The complete experiment contract and question registry are embedded, along with the supplied working paper revision 0.8. The native source, model and tokenizer remain separately pinned. The Go and compact-backbone identities are recorded before their respective measurements.

Indecis source: https://github.com/Bornholm/indecis/tree/9930c7db1913818db7cee68c1b42a384f59592e3  
Compact backbone: https://huggingface.co/hotchpotch/bekko-embedding-v1-a8m  
Go manifest: https://go.dev/dl/?mode=json  
Public SNLI: https://huggingface.co/datasets/stanfordnlp/snli  
Historical E40 evidence: https://drive.google.com/drive/folders/18mtxLUnbb45MUJYr2dc5taZrrODsA_wu

Authoring tests and mock callbacks do not validate pretrained inference, CUDA dispatch, actual upstream Go API compilation, semantic labels, calibration transfer or target-machine performance. **No new GPU/indecis benchmark results are prefilled.**